In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# BigQuery DataFrames: Interactive Analytics to Production

This notebook demonstrates the end-to-end journey of building a data pipeline in BigQuery DataFrames (`bigframes`), going from **interactive prototyping** to **production deployment** using three experimental features:

1. **Local Peek Cache**: Accelerates your interactive loop by caching a local sample during exploration, executing subsequent step-by-step inspections instantly on your client machine.
2. **Python-to-SQL UDF Transpiler**: Compiles custom Python functions into native BigQuery SQL expressions (including `groupby` aggregations and window operations) with zero deployment overhead.
3. **Productionize Mode**: Takes your validated pipeline logic and compiles it into a single optimized SQL script or a full Google Cloud Dataform project.

---

### The Story:
We will analyze a large dataset of **Chicago Taxi Trips** (200M+ rows). 
* **Prototyping**: We'll filter outliers, compute tip percentages, and perform species-level normalization using custom Python window operations. We'll leverage the **Local Peek Cache** to iterate instantly.
* **Production**: Once we are happy with the logic, we will encapsulate the pipeline in **Productionize Mode** and compile it to a production SQL script or export a full Dataform project.


## Setup & Option Configuration

We start by importing our libraries, setting our active GCP project, and enabling both the **peek cache** and the **Python transpiler** experimental options.


In [ ]:
import time
import bigframes.pandas as bpd

# Configure the BigQuery project to 'bigframes-dev' for this demo
PROJECT_ID = "bigframes-dev"
bpd.options.bigquery.project = PROJECT_ID

# Enable Local Peek Cache and Enable Python UDF Transpiler
bpd.options.compute.enable_peek_cache = True
bpd.options.compute.peek_cache_size = 10000
bpd.options.experiments.enable_python_transpiler = True
bpd.options.bigquery.ordering_mode = 'partial'

session = bpd.get_global_session()

/usr/local/google/home/tbergeron/src/google-cloud-python/packages/bigframes/bigframes/_config/experiment_options.py:182: PythonTranspilerPreviewWarning: Python transpiler is an unstable, experimental feature, and not yet
fully validated, use at your own risk.
  warnings.warn(msg, category=bfe.PythonTranspilerPreviewWarning)
/usr/local/google/home/tbergeron/src/google-cloud-python/packages/bigframes/bigframes/core/global_session.py:113: DefaultLocationWarning: No explicit location is set, so using location US for the session.
  _global_session = bigframes.session.connect(


Using Google Cloud Project: bigframes-dev
Peek cache enabled: True
Python transpiler enabled: True


## Phase 1: Prototyping & Exploration (Peek Cache & UDF Transpiler)

We load the Chicago Taxi Trips public table (200M+ rows). 

First, we calculate a small aggregated view of average fares and tips grouped by payment type. By displaying/downloading it, the result gets cached locally as a **complete cache entry** because the number of rows is small (< 10,000).


In [3]:
# Create lazy DataFrame reference
df = bpd.read_gbq("bigquery-public-data.chicago_taxi_trips.taxi_trips")

start_time = time.time()

# Group by payment type and get average fare and trip count
df_agg = df.groupby("payment_type").agg(
    fare_mean=("fare", "mean"),
    tips_mean=("tips", "mean"),
    trip_count=("fare", "count")
).reset_index()

# Download and display (this queries BigQuery once and caches the complete output)
print("Computing aggregation in BigQuery...")
df_agg_pd = df_agg.to_pandas()
print(df_agg_pd)

print(f"\nInitial query and download execution time: {time.time() - start_time:.4f} seconds")


Computing aggregation in BigQuery...


   payment_type  fare_mean  tips_mean  trip_count
0          Cash  11.639907   0.005749   119532408
1   Credit Card  16.430913   3.573094    86256408
2       Dispute  13.940387   0.009501       88949
3        Mobile  15.972737   3.161899     2602938
4     No Charge  14.576718    0.70816      826657
5         Pcard   9.502388   0.232494       36895
6        Prcard  22.824426   0.193083     2202147
7       Prepaid  22.035569   0.007782        1898
8         Split   14.39493   3.201069        3442
9       Unknown  19.505744   0.088709     1538372
10     Way2ride  12.892676   2.411479         142

Initial query and download execution time: 1.0099 seconds


### Fast Iteration with UDF Transpiler

Now, we want to perform further analysis on this aggregated result:
1. Filter out payment types with less than 1,000 trips.
2. Apply a custom Python UDF to label the average fares.
3. Compute the tip ratio.
4. Sort by the tip ratio.

Since `df_agg` is already cached locally, these steps (including UDF compilation and execution) will run **100% locally on the client** in milliseconds, completely bypassing BigQuery.


In [4]:
start_time = time.time()

# 1. Filter out rare payment types
df_common = df_agg[df_agg["trip_count"] >= 1000]

# 2. Define custom classification function
def classify_fare(mean_fare):
    if mean_fare is None:
        return "Unknown"
    elif mean_fare > 20:
        return "Expensive"
    else:
        return "Standard"

# 3. Apply the classification UDF (transpiled to CASE) and compute tip ratio
df_explored = df_common.assign(
    fare_class=df_common["fare_mean"].apply(classify_fare),
    tip_ratio=df_common["tips_mean"] / df_common["fare_mean"]
)

# 4. Sort and display
result = df_explored.sort_values(by="tip_ratio", ascending=False)
print(result.to_pandas())

print(f"\nLocal exploration execution time: {time.time() - start_time:.4f} seconds")


  payment_type  fare_mean  tips_mean  trip_count fare_class  tip_ratio
0        Split   14.39493   3.201069        3442   Standard   0.222375
1  Credit Card  16.430913   3.573094    86256408   Standard   0.217462
2       Mobile  15.972737   3.161899     2602938   Standard   0.197956
3    No Charge  14.576718    0.70816      826657   Standard   0.048582
4        Pcard   9.502388   0.232494       36895   Standard   0.024467
5       Prcard  22.824426   0.193083     2202147  Expensive   0.008459
6      Unknown  19.505744   0.088709     1538372   Standard   0.004548
7      Dispute  13.940387   0.009501       88949   Standard   0.000682
8         Cash  11.639907   0.005749   119532408   Standard   0.000494
9      Prepaid  22.035569   0.007782        1898  Expensive   0.000353

Local exploration execution time: 0.0385 seconds


## Phase 2: Compiling to Production (Productionize Mode)

Once you have verified the pipeline logic on a local sample, you can prepare the pipeline for production deployment.

We will:
1. Enter the `productionize` context block.
2. Define a query parameter `min_trip_count` to filter out groupings dynamically.
3. Re-run our data pipeline logic.
4. Record writing the results to a target BigQuery table (`bigframes_productionize_demo.payment_type_analysis`).
5. Compile the entire data flow into a clean SQL script or export a Dataform project repository.


In [5]:
# Create the target dataset
DATASET_ID = "bigframes_productionize_demo"
session.bqclient.create_dataset(DATASET_ID, exists_ok=True)

with bpd.productionize() as pipeline:
    # 1. Define query parameter for trip count limit
    min_trip_count = bpd.parameter("min_trip_count", dtype=int)

    # 2. Lazy data loading
    df_raw = bpd.read_gbq("bigquery-public-data.chicago_taxi_trips.taxi_trips")

    # 3. Groupby Aggregation
    df_grouped = df_raw.groupby("payment_type").agg(
        fare_mean=("fare", "mean"),
        tips_mean=("tips", "mean"),
        trip_count=("fare", "count")
    ).reset_index()

    # 4. Filter using the parameter
    df_filtered_trips = df_grouped[df_grouped["trip_count"] >= min_trip_count]

    # 5. Define and apply the classification UDF
    # We specify the target dataset for persistent UDF registration
    @bpd.udf(input_types=[float], output_type=str, dataset=DATASET_ID, name="classify_fare")
    def classify_fare(mean_fare):
        if mean_fare is None:
            return "Unknown"
        elif mean_fare > 20:
            return "Expensive"
        else:
            return "Standard"

    df_final = df_filtered_trips.assign(
        fare_class=df_filtered_trips["fare_mean"].apply(classify_fare),
        tip_ratio=df_filtered_trips["tips_mean"] / df_filtered_trips["fare_mean"]
    )

    df_selected = df_final[["payment_type", "fare_mean", "tips_mean", "trip_count", "fare_class", "tip_ratio"]]

    # 6. Record the write operation to the destination table
    df_selected.to_gbq(
        f"{PROJECT_ID}.{DATASET_ID}.payment_type_analysis",
        if_exists="replace"
    )

print("Pipeline recorded successfully!")


/usr/local/google/home/tbergeron/src/google-cloud-python/packages/bigframes/bigframes/core/logging/log_adapter.py:183: TimeTravelCacheWarning: Reading cached table from 2026-07-15 23:36:45.769604+00:00 to avoid
incompatibilies with previous reads of this table. To read the latest
version, set `use_cache=False` or close the current session with
Session.close() or bigframes.pandas.close_session().
  return method(*args, **kwargs)
/usr/local/google/home/tbergeron/src/google-cloud-python/packages/bigframes/bigframes/pandas/__init__.py:212: PreviewWarning: udf is in preview.
  return global_session.with_default_session(


Pipeline recorded successfully!


### Output 1: Compiling to a BigQuery SQL script

Let's retrieve the compiled SQL script. This script packages the entire logical execution plan (including the creation of the persistent UDF) and can be scheduled directly in BigQuery.


In [6]:
sql_script = pipeline.to_sql()
print(sql_script)


CREATE OR REPLACE FUNCTION `bigframes-dev.bigframes_productionize_demo.classify_fare`(mean_fare FLOAT64)
RETURNS STRING
LANGUAGE python

OPTIONS (runtime_version='python-3.11', entry_point='bigframes_handler')
AS r'''

def classify_fare(mean_fare):
    if mean_fare is None:
        return "Unknown"
    elif mean_fare > 20:
        return "Expensive"
    else:
        return "Standard"


def bigframes_handler(*args):
    return classify_fare(*args)

''';

CREATE OR REPLACE TABLE `bigframes-dev.bigframes_productionize_demo.payment_type_analysis` AS
SELECT `bfuid_col_43` AS `payment_type`, `bfuid_col_44` AS `fare_mean`, `bfuid_col_45` AS `tips_mean`, `bfuid_col_46` AS `trip_count`, `bfuid_col_47` AS `fare_class`, `bfuid_col_48` AS `tip_ratio` FROM (SELECT
  *
FROM (
  SELECT
    `t3`.`payment_type`,
    `t3`.`bfuid_col_25`,
    `t3`.`bfuid_col_26`,
    `t3`.`bfuid_col_27`,
    `t3`.`bfuid_col_29`,
    `t3`.`bfuid_col_30`,
    `t3`.`bfuid_col_31`,
    `t3`.`bfuid_col_32`,
    `t3`.`bfuid_c

### Output 2: Exporting to a Dataform Project

Alternatively, if you manage your pipelines in Dataform, you can export the compiled pipeline as a local Dataform project. This generates a structured repository with SQLX files, automatically tracing the UDF and table dependency hierarchy.


In [7]:
import os
import tempfile

export_dir = os.path.join(tempfile.gettempdir(), "dataform_taxi_pipeline")
os.makedirs(export_dir, exist_ok=True)

# Export the project
print(f"Exporting pipeline to: {export_dir}")
pipeline.export_dataform(export_dir)

# List generated Dataform definitions
print("\nGenerated definitions files:")
for root, _, files in os.walk(os.path.join(export_dir, "definitions")):
    for file in files:
        print(f"- definitions/{file}")


Exporting pipeline to: /tmp/dataform_taxi_pipeline

Generated definitions files:
- definitions/payment_type_analysis.sqlx
- definitions/classify_fare.sqlx
